In [27]:
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
from typing import TypedDict, Literal, Annotated
from langchain.messages import SystemMessage, HumanMessage
import operator

In [28]:
generate_llm = ChatOllama(model="llama3.2:1b")
evalute_llm = ChatOllama(model="llama3.2:1b")
optimize_llm = ChatOllama(model="llama3.2:1b")


In [29]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [30]:
structured_evaluator_llm = evalute_llm.with_structured_output(TweetEvaluation)

In [31]:
class tweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]

In [32]:
def generate_tweet(state: tweetState):

    # prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    # send generator_llm
    response = generate_llm.invoke(messages).content

    # return response
    return {'tweet': response, 'tweet_history': [response]}

In [33]:
def evaluate_tweet(state: tweetState):

    # prompt
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    response = structured_evaluator_llm.invoke(messages)

    return {'evaluation':response.evaluation, 'feedback': response.feedback, 'feedback_history': [response.feedback]}

In [34]:
def optimize_tweet(state: tweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = optimize_llm.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}

In [35]:
def route_evaluation(state: tweetState):

    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [36]:
graph = StateGraph(tweetState)

graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate_tweet", evaluate_tweet)
graph.add_node("optimize_tweet", optimize_tweet)

graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet", route_evaluation, {"approved" : END, "needs_improvement": "optimize_tweet"})

workflow = graph.compile()


In [38]:
initial_state = {
    "topic": "srhberhb",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)

result

{'topic': 'srhberhb',
 'tweet': '"Tried to make a sandwich, but my fridge is now a crime scene. \'Srhberhb\' is on the fridge, counter, and I\'m pretty sure my cat \'s\' if I feed it. Guess you could say it\'s a real \'berb\' situation #srhberhb"',
 'evaluation': 'needs_improvement',
 'feedback': "This tweet is a rehashed version of a common meme, with the subject being 'srhberhb' having appeared on a fridge and counter multiple times. The humor is also familiar and doesn't bring anything new to the table. The punchline feels like a lazy way to drive home the absurdity, and the ",
 'iteration': 2,
 'max_iteration': 5,
 'tweet_history': ['"I was trying to make a simple sandwich, and now my fridge is a crime scene. I\'ve got \'srhberhb\' on the fridge, \'srhberhb\' on the counter, and I\'m pretty sure my cat \'s\' if I feed it"',
  '"Tried to make a sandwich, but my fridge is now a crime scene. \'Srhberhb\' is on the fridge, counter, and I\'m pretty sure my cat \'s\' if I feed it. Guess 